# Complainify AI — 08 : ML Production Data Lifecycle

How the project turns raw complaints into a versioned, production-ready model. This notebook mirrors the real `ml/` package (`retrain.py`, `validate_data.py`, `model_registry.py`) using the actual data files.

**The lifecycle at a glance:**
1. Students submit complaints → stored in MySQL (new data lives in the DB, never auto-trained)
2. Admin confirms the true category on the complaint detail page (`category_confirmed=1`)
3. Retrain (manual button or scheduled) merges CSV + confirmed DB rows
4. Validation gate rejects bad/duplicate rows before they may train
5. New valid rows are appended to the collected `train_dataset.csv`
6. Model is trained, evaluated, and saved as a NEW VERSION in the registry
7. The active model pointer flips; the old model stays archived for comparison

In [1]:
# Step 1 — the collected dataset is the single source of truth
import csv, os

DATA_PATH = r'E:/Project-VI/workspace/ComplaintMgmtSystem/data/train_dataset.csv'
with open(DATA_PATH, encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
print('Collected rows      :', len(rows))
print('Columns             :', list(rows[0].keys()))
print('Unique texts        :', len({r['text'] for r in rows}))
print('Sample row          :', dict(list(rows[0].items())))


Collected rows      : 7395
Columns             : ['text', 'category', 'priority', 'source', 'category_encoded', 'priority_encoded']
Unique texts        : 7210
Sample row          : {'text': 'ventilator inverter exhaust noisy balcony', 'category': 'Hostels', 'priority': 'High', 'source': 'synth_highacc', 'category_encoded': '1', 'priority_encoded': '2'}


## The Validation Gate

Before any row may enter training it must pass `validate_data.validate_rows()`:
empty text, too-short, unknown category, duplicates (exact-text) and gibberish are rejected with a reason. A JSON report is written to `ml/reports/`.

In [2]:
import sys
sys.path.insert(0, r'E:/Project-VI/workspace/ComplaintMgmtSystem/ml')
from validate_data import validate_rows

# Simulate the batch of admin-confirmed complaints about to enter training
candidates = [
    {'text': 'Hostel water supply stopped in block C since morning', 'category': 'Hostels'},
    {'text': 'WiFi keeps disconnecting in the library',               'category': 'IT Support'},
    {'text': '',                                                      'category': 'Canteen'},
    {'text': 'aaaaaaaaaaaa',                                          'category': 'Hostels'},
    {'text': 'Hostel water supply stopped in block C since morning', 'category': 'Hostels'},
    {'text': 'Not a real category',                                  'category': 'Space'},
]

result = validate_rows(candidates, existing_texts=[])
print('valid   :', len(result['valid']))
print('rejected:', len(result['rejected']))
for r in result['rejected']:
    print('  -', r['reason'], '->', (r['row'].get('text') or '')[:50])
print('report written to:', result['report_path'])


valid   : 2
rejected: 4
  - empty_text -> 
  - too_short -> aaaaaaaaaaaa
  - duplicate_text -> Hostel water supply stopped in block C since morni
  - unknown_category -> Not a real category
report written to: E:\Project-VI\workspace\ComplaintMgmtSystem\ml\reports\validation_report_20260807_200210.json


## Versioned Model Registry

`ml/model_registry.py` keeps every trained model as `ml/models/model_<version>.json`
plus a `manifest.json` with accuracy / F1 / sample counts per version. The active version is used for predictions; older versions stay for comparison.

In [3]:
from model_registry import list_versions, latest_version_id, load_latest, active_metrics

print('Active version :', latest_version_id())
for v in list_versions():
    m = v.get('metrics', {})
    acc = f"{m.get('accuracy')}%" if m.get('accuracy') is not None else 'N/A'
    print(f"  {v['id']}  saved={v['saved_at']}  acc={acc}  f1={m.get('macro_f1')}")

model = load_latest()
print('\nLoaded active model :', model.vocab_size, 'vocab tokens,', len(model.classes), 'classes')


Active version : v20260807_200111
  v20260807_200111  saved=2026-08-07 20:01:11  acc=96.67%  f1=0.9675
  v20260807_200043  saved=2026-08-07 20:00:43  acc=96.53%  f1=0.9657
  v20260807_200002  saved=2026-08-07 20:00:02  acc=96.19%  f1=0.9624
  v20260807_195723  saved=2026-08-07 19:57:23  acc=95.77%  f1=0.9586

Loaded active model : 2324 vocab tokens, 10 classes


## Triggering a Retrain

Two ways (from the docs):
- **Manual** — admin clicks *Retrain Model* on the Training page (runs `ml/retrain.py`)
- **Scheduled** — set `RETRAIN_SCHEDULE_HOURS` in `.env` (e.g. `168` = weekly); the Flask app starts a background thread that retrains on that interval

Retrain flow: collect CSV + `category_confirmed=1` DB rows → validate → append new unique rows to the CSV → fit → evaluate 80/20 → `model_registry.save_version()` → flip active pointer → update `ml/training_log.json`.

Run it directly (DB must be running for confirmed rows to be included):

In [4]:
import subprocess, sys
proc = subprocess.run([sys.executable, r'E:/Project-VI/workspace/ComplaintMgmtSystem/ml/retrain.py'],
                     capture_output=True, text=True, timeout=600)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-500:])


{"last_version": "v20260807_200210", "last_trained": "2026-08-07 20:02:10", "train_samples": 5768, "test_samples": 1442, "accuracy": 96.32, "note": null, "macro_f1": 0.9638, "per_class": [{"category": "IT Support", "precision": 0.9792, "recall": 0.9792, "f1": 0.9792, "samples": 144}, {"category": "Hostels", "precision": 0.933, "recall": 0.9489, "f1": 0.9408, "samples": 176}, {"category": "Academics", "precision": 0.962, "recall": 0.9441, "f1": 0.953, "samples": 161}, {"category": "Fees / Finance", "precision": 0.9592, "recall": 0.9527, "f1": 0.9559, "samples": 148}, {"category": "Maintenance", "precision": 0.9247, "recall": 0.9441, "f1": 0.9343, "samples": 143}, {"category": "Transport", "precision": 0.9655, "recall": 1.0, "f1": 0.9825, "samples": 140}, {"category": "Security / Discipline", "precision": 0.9921, "recall": 0.9328, "f1": 0.9615, "samples": 134}, {"category": "Administration", "precision": 0.9421, "recall": 0.9344, "f1": 0.9383, "samples": 122}, {"category": "Library", "pr

## Verify: new version registered

After the retrain above the registry should show a second, newer version.

In [5]:
from model_registry import list_versions, latest_version_id
print('Active version :', latest_version_id())
print('Total versions :', len(list_versions()))
for v in list_versions():
    print('  ', v['id'], v['saved_at'])


Active version : v20260807_200210
Total versions : 5
   v20260807_200210 2026-08-07 20:02:10
   v20260807_200111 2026-08-07 20:01:11
   v20260807_200043 2026-08-07 20:00:43
   v20260807_200002 2026-08-07 20:00:02
   v20260807_195723 2026-08-07 19:57:23


## API surface (backend/app.py)

- `POST /api/predict` — categorize + `model_version` used
- `POST /api/retrain` — trigger the pipeline (JSON result)
- `GET /api/models/latest` — active model info + metrics
- `GET /api/models` — all versions for comparison
- Student submit stores the auto-prediction + `model_version`; the admin `Confirm for Training` button sets `category_confirmed=1` — the only path for new data into training.

**Key rule:** the model never trains on its own auto-predicted labels.